In [1]:
pip install faker

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   ----- -------------------------

In [2]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta

fake = Faker("en_IN")
np.random.seed(42)
Faker.seed(42)

In [3]:
n = 5000

In [4]:
df = pd.DataFrame()

In [5]:
df["customerid"] = range(100001,100001+n)

In [6]:
df["customer_name"] = [fake.name() for _ in range(n)]

In [7]:
df["gender"] = np.random.choice(
    ["Male","Female"],
    n,
    p=[0.52,0.48]
)

In [8]:
states = [
    "Maharashtra",
    "Karnataka",
    "Delhi",
    "Gujarat",
    "Tamil Nadu",
    "Telangana",
    "Uttar Pradesh",
    "Rajasthan"
]

df["state"] = np.random.choice(
    states,
    n,
    p=[0.22,0.15,0.12,0.12,0.10,0.10,0.10,0.09]
)

In [9]:
df["country"] = "India"

In [10]:
start = datetime(1965,1,1)
end = datetime(2005,12,31)

df["dob"] = [
    start + timedelta(
        days=np.random.randint(
            0,
            (end-start).days
        )
    )
    for _ in range(n)
]

In [11]:
df.head()
df.shape

(5000, 6)

In [12]:
# Plan Type
df["plan_type"] = np.random.choice(
    ["Basic", "Standard", "Premium"],
    n,
    p=[0.45, 0.35, 0.20]
)

# Contract Type
df["contract_type"] = np.random.choice(
    ["Monthly", "Annual"],
    n,
    p=[0.65, 0.35]
)

# Subscription Type
df["subscription_type"] = np.where(
    df["plan_type"] == "Premium",
    "Premium",
    "Regular"
)

In [13]:
df.head()

,customerid,customer_name,gender,state,country,dob,plan_type,contract_type,subscription_type
0,100001,Aryan Maharaj,Male,Delhi,India,1996-05-04,Premium,Monthly,Premium
1,100002,Udant Dewan,Female,Delhi,India,1967-05-14,Standard,Monthly,Regular
2,100003,Gagan Sami,Female,Uttar Pradesh,India,2005-10-01,Standard,Monthly,Regular
3,100004,Ayushman Chander,Female,Karnataka,India,1998-04-03,Standard,Monthly,Regular
4,100005,Viraj Tiwari,Male,Uttar Pradesh,India,1995-12-10,Standard,Annual,Regular


In [14]:
# Monthly Charges
df["monthly_charges"] = np.select(
    [
        df["plan_type"] == "Basic",
        df["plan_type"] == "Standard",
        df["plan_type"] == "Premium"
    ],
    [
        np.random.randint(299, 499, n),
        np.random.randint(500, 899, n),
        np.random.randint(900, 1499, n)
    ]
)

In [15]:
df.groupby("plan_type")["monthly_charges"].describe()

,count,mean,std,min,25%,50%,75%,max
plan_type,,,,,,,,
Basic,2209.0,396.649615,58.199319,299.0,346.00,396.0,447.00,498.0
Premium,1012.0,1197.272727,173.226125,900.0,1043.75,1195.5,1342.25,1498.0
Standard,1779.0,692.462619,116.795516,500.0,589.00,689.0,795.00,898.0


In [16]:
# CSAT Score (1-5)
df["csat_score"] = np.random.choice(
    [1, 2, 3, 4, 5],
    n,
    p=[0.08, 0.12, 0.20, 0.35, 0.25]
)

# Escalations
df["escalations"] = np.random.choice(
    [0, 1, 2, 3],
    n,
    p=[0.60, 0.25, 0.10, 0.05]
)

# Complaint Count
df["complaint_count"] = np.random.poisson(
    lam=1,
    size=n
)

In [17]:
df[["csat_score","escalations","complaint_count"]].describe()

,csat_score,escalations,complaint_count
count,5000.000000,5000.000000,5000.000000
mean,3.552400,0.625400,1.002400
std,1.211094,0.889063,0.997694
min,1.000000,0.000000,0.000000
25%,3.000000,0.000000,0.000000
50%,4.000000,0.000000,1.000000
75%,4.000000,1.000000,2.000000
max,5.000000,3.000000,6.000000


In [18]:
# Base churn score
df["churn_score"] = 20

# Low CSAT increases churn score
df.loc[df["csat_score"] <= 2, "churn_score"] += 40

# Escalations increase churn score
df["churn_score"] += df["escalations"] * 10

# Monthly contract increases churn risk
df.loc[df["contract_type"] == "Monthly", "churn_score"] += 15

# Premium plan customers churn less
df.loc[df["plan_type"] == "Premium", "churn_score"] -= 10

# Add randomness
df["churn_score"] += np.random.randint(-10, 11, n)

# Keep score between 0 and 100
df["churn_score"] = df["churn_score"].clip(0, 100)

In [19]:
df["churn_flag"] = np.where(df["churn_score"] >= 60, 1, 0)

In [20]:
df["churn_flag"].value_counts()

churn_flag
0    3996
1    1004
Name: count, dtype: int64

In [21]:
df.groupby("contract_type")["churn_flag"].mean()*100

contract_type
Annual     12.953930
Monthly    24.247227
Name: churn_flag, dtype: float64

In [22]:
# Customer Lifetime Value
df["cltv"] = (
    df["monthly_charges"] *
    np.random.randint(12, 61, n)
).round(2)

In [23]:
df.loc[df["plan_type"]=="Premium","cltv"] *= 1.30

C:\Users\USER\AppData\Local\Temp\ipykernel_20336\4117217997.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[24859.9 68538.6 31738.2 ... 76284.  37258.  57142.8]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df.loc[df["plan_type"]=="Premium","cltv"] *= 1.30


In [24]:
df[["monthly_charges","cltv"]].head()

,monthly_charges,cltv
0,1471,24859.9
1,687,22671.0
2,823,11522.0
3,834,42534.0
4,708,41064.0


In [25]:
from datetime import datetime

# Subscription Start Date
start = pd.to_datetime("2022-01-01")
end = pd.to_datetime("2024-12-31")

df["subscription_start_date"] = pd.to_datetime(
    np.random.randint(
        start.value//10**9,
        end.value//10**9,
        n
    ),
    unit="s"
)

# Renewal Date
df["renewal_date"] = df["subscription_start_date"] + pd.to_timedelta(
    np.random.randint(30, 366, n),
    unit="D"
)

In [26]:
reasons = [
    "High Pricing",
    "Poor Service",
    "Competitor",
    "Network Issues",
    "Relocation"
]

df["cancellation_date"] = pd.NaT

mask = df["churn_flag"] == 1

df.loc[mask, "cancellation_date"] = (
    df.loc[mask, "subscription_start_date"] +
    pd.to_timedelta(
        np.random.randint(30, 365, mask.sum()),
        unit="D"
    )
)

df["cancellation_reason"] = np.where(
    df["churn_flag"] == 1,
    np.random.choice(reasons, n),
    "Active"
)

In [27]:
df.head()

,customerid,customer_name,gender,state,country,dob,plan_type,contract_type,subscription_type,monthly_charges,csat_score,escalations,complaint_count,churn_score,churn_flag,cltv,subscription_start_date,renewal_date,cancellation_date,cancellation_reason
0,100001,Aryan Maharaj,Male,Delhi,India,1996-05-04,Premium,Monthly,Premium,1471,5,0,0,35,0,24859.9,2024-11-22 01:37:15,2025-02-09 01:37:15,NaT,Active
1,100002,Udant Dewan,Female,Delhi,India,1967-05-14,Standard,Monthly,Regular,687,5,0,1,42,0,22671.0,2022-06-18 22:15:15,2022-10-22 22:15:15,NaT,Active
2,100003,Gagan Sami,Female,Uttar Pradesh,India,2005-10-01,Standard,Monthly,Regular,823,1,3,1,95,1,11522.0,2024-01-02 00:34:28,2024-10-09 00:34:28,2024-06-21 00:34:28,Poor Service
3,100004,Ayushman Chander,Female,Karnataka,India,1998-04-03,Standard,Monthly,Regular,834,4,0,0,37,0,42534.0,2022-10-22 21:14:11,2023-01-22 21:14:11,NaT,Active
4,100005,Viraj Tiwari,Male,Uttar Pradesh,India,1995-12-10,Standard,Annual,Regular,708,5,0,1,23,0,41064.0,2022-05-24 18:18:15,2022-09-16 18:18:15,NaT,Active


In [28]:
df = df[
[
'customerid',
'customer_name',
'gender',
'country',
'state',
'dob',
'subscription_start_date',
'subscription_type',
'renewal_date',
'plan_type',
'contract_type',
'cancellation_date',
'cancellation_reason',
'monthly_charges',
'cltv',
'churn_score',
'churn_flag',
'escalations',
'csat_score',
'complaint_count'
]
]

In [29]:
print(df.shape)

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicates")
print(df.duplicated().sum())

print("\nData Types")
print(df.dtypes)

(5000, 20)

Missing Values
customerid                    0
customer_name                 0
gender                        0
country                       0
state                         0
dob                           0
subscription_start_date       0
subscription_type             0
renewal_date                  0
plan_type                     0
contract_type                 0
cancellation_date          3996
cancellation_reason           0
monthly_charges               0
cltv                          0
churn_score                   0
churn_flag                    0
escalations                   0
csat_score                    0
complaint_count               0
dtype: int64

Duplicates
0

Data Types
customerid                          int64
customer_name                      object
gender                             object
country                            object
state                              object
dob                        datetime64[ns]
subscription_start_date    datetime64[ns]


In [30]:
premium = df["plan_type"]=="Premium"

df.loc[premium,"csat_score"] = np.random.choice(
    [3,4,5],
    premium.sum(),
    p=[0.10,0.35,0.55]
)

In [31]:
high_churn = df["churn_score"]>=70

df.loc[high_churn,"complaint_count"] = np.random.randint(
    2,
    6,
    high_churn.sum()
)

In [32]:
df.loc[
    (df["csat_score"]<=2) &
    (df["churn_flag"]==1),
    "cancellation_reason"
] = "Poor Service"

df.loc[
    (df["monthly_charges"]>1000) &
    (df["churn_flag"]==1),
    "cancellation_reason"
] = "High Pricing"

In [34]:
import os

os.makedirs("data/simulated", exist_ok=True)

df.to_csv(
    "data/simulated/customer_churn_5000.csv",
    index=False
)

In [35]:
df.shape

(5000, 20)